In [ ]:
from pathlib import Path
import pandas as pd

import pyam
import nomenclature

In [ ]:
project_id = "CEMICS"
project_name = "CEMICS"

In [ ]:
ar6_db = pyam.iiasa.Connection("ar6-public")

In [ ]:
runs = ar6_db.properties().reset_index()

In [ ]:
list(runs.scenario.unique())

In [ ]:
df = ar6_db.query(
    variable="Carbon Sequestration|CCS",
    scenario=["R_*"],
)

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i,
                (
                    i.replace("CEMICS_", "CEMICS-")
                    .replace("1.5", "1.5°C")
                    .replace("1p5C", "1.5°C")
                    .replace("1p5", "1.5°C")
                    .replace("2.0", "2.0°C")
                    .replace("1p5C", "1.5°C")
                    .replace("1p5", "1.5°C")
                    .replace("2C", "2.0°C")
                    .replace("opt", "Optimal")
                )
            )
            for i in df.scenario if "CEMICS" in i
        ]
    ),
    inplace=True,
)

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i,
                (
                    "SDI-" + i.replace("C", "°C").replace("WB", "well-below-")
                )
            )
            for i in df.scenario
        ]
    ),
    inplace=True,
)

In [ ]:
df

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i,
                (
                    i.replace("CO_", "COMMIT-")
                    .replace("CurPol", "Current-Policies")
                    .replace("2Deg", "2°C-")
                    .replace("BAU", "Baseline")
                    .replace("_notax", "-No-Tax")
                    .replace("_2050convergence", "-2050-Convergence")
                )
            )
            for i in df.scenario
        ]
    ),
    inplace=True,
)

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i,
                (
                    i.replace("R_", "Deep-Mitigation-")
                    .replace("BAU", "Baseline")
                    .replace("SSP_", "Deep-Mitigation-")
                )
            )
            for i in df.scenario
        ]
    ),
    inplace=True,
)

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i,
                (
                    i.replace("CD-LINKS_", "CD-LINKS-")
                    .replace("NoPolicy", "No-Policy")
                )
            )
            for i in df.scenario
        ]
    ),
    inplace=True,
)

In [ ]:
df.scenario

In [ ]:
df.region

In [ ]:
region_mapping = {
  "Asian countries except Japan": "Asia (R5)",
  "Latin American countries": "Latin America (R5)",
  "Countries of the Middle East and Africa": "Middle East & Africa (R5)",
  "OECD90 and EU (and EU candidate) countries": "OECD & EU (R5)",
  "Countries from the Reforming Economies of the Former Soviet Union": "Reforming Economies (R5)",
  "Countries of Latin America and the Caribbean": "Latin America (R10)",
  "Countries of South Asia; primarily India": "India+ (R10)",
  "Countries of Sub-Saharan Africa": "Africa (R10)",
  "Countries of centrally-planned Asia; primarily China": "China+ (R10)",
  "Countries of the Middle East; Iran, Iraq, Israel, Saudi Arabia, Qatar, etc.": "Middle East (R10)",
  "Eastern and Western Europe (i.e., the EU28)": "Europe (R10)",
  "North America; primarily the United States of America and Canada": "North America (R10)",
  "Pacific OECD": "Pacific OECD (R10)",
  "Reforming Economies of Eastern Europe and the Former Soviet Union; primarily Russia": "Reforming Economies (R10)",
  "Other countries of Asia": "Rest of Asia (R10)",
  "Rest of the World (R10)": "Other (R10)",
  "United States of America": "United States", 
  "Russia": "Russian Federation",
}

In [ ]:
df.rename(region=region_mapping, inplace=True)

In [ ]:
df.filter(region="*(R6)", keep=False, inplace=True)

In [ ]:
df.filter(region=["Colombia", "Pakistan", "Taiwan", "European Union (28 member countries)"], keep=False, inplace=True)

In [ ]:
df.region

In [ ]:
#df = df.filter(region=list(region_mapping.keys()) + ['World'])

In [ ]:
definition = nomenclature.DataStructureDefinition("../../common-definitions/definitions/")

In [ ]:
definition.validate(df, dimensions=["region"])

In [ ]:
# update carbon-management variables
carbon_management_mapping = {
    "Carbon Sequestration|CCS": "Carbon Capture|Geological Storage",
}
df.rename(variable=carbon_management_mapping, inplace=True)

In [ ]:
definition.validate(df)

In [ ]:
import ixmp4
platform = ixmp4.Platform("scenariocompass-dev")

In [ ]:
for model, scenario in df.index:
    try:
        run = platform.runs.get(model=model, scenario=scenario)
        run.iamc.add(df.filter(model=model, scenario=scenario).data)
        print(f"Done {model} | {scenario}")
    except:
        print(f"SKIPPED {model} | {scenario}")

In [ ]:
model

In [ ]:
df.model